<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Control complexity

**[C05-M05-L03 · Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/)** · C05, Machine Learning: From Problem to Reliable Model · Module 5, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** recognise underfitting and overfitting from the train and CV scores, choose hyperparameters (settings chosen before training) such as `max_depth`, `min_samples_leaf` and `C` with cross-validation, and read a learning curve.

| | |
|---|---|
| **Time** | About 60 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Trees and `evaluate()`, from [Decision trees](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/). Cross-validation, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#cross-validation). |
| **You do not need** | The local project. The setup below downloads the data, writes `ticket_model.py` and writes your experiment log so far. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M05-L03-control-complexity/control-complexity-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: ticket_model.py

The next cell writes `ticket_model.py` as you wrote it in [Preprocess with a pipeline](https://learning.nextia-ai.com/courses/ml/m04/preprocess-with-a-pipeline/#write-ticket_modelpy): the column lists, `load()` and `build_pipeline()`. Run it. You should see `Writing ticket_model.py` (or `Overwriting`).

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 4, lesson 3 (Preprocess with a pipeline).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

## Setup: the experiment log

The next cell defines two helpers for your **experiment log**, the file `experiment_log.csv` in `ticket-model/`. You start the log in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log).

- `log_experiment(id, change, cv_ap, cv_std, valid_ap, decision, reason)` adds one row, with today's date. If the id is already in the log, it replaces that row, so you can run a cell again.
- `show_log()` shows the log as a table.

It also writes the 8 rows that you added in the earlier lessons (E01 to E08), so this notebook works without them. If you keep your own log in the local project, skip the last line of the cell.

> **Check.** You should see the log with 8 rows, from E01 to E08.

In [ ]:
# The experiment log: one row per experiment, in experiment_log.csv.
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "valid_ap", "decision", "reason"]


def log_experiment(id, change, cv_ap, cv_std, valid_ap=None, decision="", reason=""):
    """Add one row to the log (or replace the row with the same id)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_ap": round(float(cv_ap), 3), "cv_std": round(float(cv_std), 3),
           "valid_ap": None if valid_ap is None else round(float(valid_ap), 3),
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)


def show_log(columns=("id", "change", "cv_ap", "cv_std", "valid_ap", "decision")):
    """The log as a table (without the date and the reason, unless you ask for them)."""
    log = pd.read_csv(LOG_FILE, dtype={"date": str}).sort_values("id")
    with pd.option_context("display.max_colwidth", 60, "display.width", 200):
        print(log[list(columns)].fillna("").to_string(index=False))


# The rows of the earlier lessons (id, change, cv_ap, cv_std, valid_ap, decision, reason).
EARLIER_ROWS = [
    ('E01', 'Baseline: one score for every ticket (the train rate)', 0.117, 0.000, 0.155, 'baseline', 'The floor: any useful model must be above it.'),
    ('E02', "Baseline: the rule 'flag priority 1', as a 0/1 score", 0.178, 0.008, 0.228, 'baseline', 'What Larkfield does today: the model must beat it.'),
    ('E03', 'Logistic regression, the 10 C04 features', 0.327, 0.012, 0.353, 'keep', 'Far above both baselines; train AP 0.328, so it does not overfit.'),
    ('E04', "E03 with class_weight='balanced'", 0.323, 0.012, None, 'reject', 'Lower than E03 on all 5 folds; one more setting for no gain.'),
    ('E05', "Baseline: the rule 'flag priority 1 or team payment'", 0.172, 0.006, None, 'reject', 'Lower than the simpler rule E02.'),
    ('E06', 'Decision tree, max_depth=3', 0.234, 0.011, None, 'reject', 'Far below E03.'),
    ('E07', 'Decision tree, max_depth=6', 0.258, 0.010, None, 'reject', 'The best of the three trees, still far below E03.'),
    ('E08', 'Decision tree, no depth limit', 0.137, 0.004, None, 'reject', 'It memorises train (train AP 1.000) and fails on new tickets.'),
]
for row in EARLIER_ROWS:
    log_experiment(*row)
show_log()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Too simple and too complex

A model can learn too little, like a student who knows one rule for everything, or too much of the wrong thing, like a student who learns the practice answers by heart. Two scores tell them apart: the score on the train tickets and the CV score.

The next cell loads the train set and defines the same helpers as [Decision trees](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/): `cv` (the 5 folds), `tree_pipeline()` and `evaluate()`. Run it.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.tree import DecisionTreeClassifier, export_text

from ticket_model import FEATURES, TARGET, build_pipeline, load

train = load("train")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def tree_pipeline(model):
    """build_pipeline() without the scaler: a tree does not need it."""
    return build_pipeline(model).set_params(prepare__numbers__scale="passthrough")


def evaluate(pipeline):
    """Train AP (fit on all train tickets) and CV AP (mean ± std of the 5 folds)."""
    fold_scores = cross_val_score(pipeline, train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
    pipeline.fit(train[FEATURES], train[TARGET])
    train_ap = average_precision_score(train[TARGET], pipeline.predict_proba(train[FEATURES])[:, 1])
    return {"train AP": round(float(train_ap), 3), "CV AP": round(float(fold_scores.mean()), 3),
            "std": round(float(fold_scores.std()), 3)}


print(len(train), "train tickets")

> **Check.** You should see `16939 train tickets`.

Start with the 10 tickets of the lesson *Decision trees*. The next cell grows three trees on them: 1 question deep, 2 deep, and with no limit (`max_depth=None`). It prints the number of leaves and each ticket's score, then the questions of the last tree.

> **Predict.** How many leaves does the tree with no limit need to score every ticket correctly?

In [ ]:
tiny = pd.DataFrame({
    "ticket": [f"T-{i:02d}" for i in range(1, 11)],
    "priority": [1, 1, 1, 1, 2, 2, 2, 3, 3, 3],
    "word_count": [150, 40, 120, 45, 130, 160, 20, 30, 145, 60],
    "escalated": [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
})
x_tiny = tiny[["priority", "word_count"]]
for depth in (1, 2, None):
    tiny_tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(x_tiny, tiny["escalated"])
    print(f"max_depth={depth}: {tiny_tree.get_n_leaves()} leaves, scores", tiny_tree.predict_proba(x_tiny)[:, 1].round(2))
print(export_text(tiny_tree, feature_names=["priority", "word_count"]))

> **Check.** You should see 2, 4 and 5 leaves. With no limit, the scores are exactly the targets: `[1. 0. 1. 1. 1. 0. 0. 0. 0. 0.]`. The last tree has a third question, `word_count <= 137.50`.

The third question separates T-05 (130 words, escalated) from T-06 (160) and T-09 (145). The rule it makes: "a priority 2 or 3 ticket with 96 to 137 words escalates, with 138 or more it does not". That rule comes from one ticket. It will not repeat in new tickets.

- **Overfitting:** the model learns details of the train tickets that do not repeat in new tickets. Train scores are high, scores on new tickets are lower.
- **Underfitting:** the model is too simple to learn the pattern that is there. Both scores are low.

## 2. The tree with no limit, on the real data

> **Predict.** In *Decision trees*, this tree had train AP 1.000. How many leaves does it need for 16,939 tickets? Run the cell.

In [ ]:
full_tree = tree_pipeline(DecisionTreeClassifier(random_state=0))
print("tree, no limit:", evaluate(full_tree))
model = full_tree.named_steps["model"]
print("leaves:", model.get_n_leaves(), " depth:", model.get_depth())

> **Check.** You should see `train AP 1.0`, `CV AP 0.137`, `std 0.004`, then `leaves: 2210  depth: 31`.

2,210 leaves, and every one is pure, so every train ticket gets exactly its own target. On new tickets, the tree is only a little above the floor of 0.117.

## 3. Bias and variance

Two words describe the two failures:

- **Bias:** the error of a model that is too simple. It misses the same part of the pattern, whatever train tickets it gets.
- **Variance:** how much the model changes when its train tickets change.

The next cell fits trees on five different random halves of train, and scores one May ticket, T-117529, which escalated. It also measures the spread of the five scores for every May ticket, and prints the mean. Run it.

In [ ]:
valid = load("valid")
ticket = valid[valid["ticket_id"] == "T-117529"]
for depth in (None, 3):
    ticket_scores, all_scores = [], []
    for seed in range(5):
        half = train.sample(frac=0.5, random_state=seed)
        pipeline = tree_pipeline(DecisionTreeClassifier(max_depth=depth, random_state=0))
        pipeline.fit(half[FEATURES], half[TARGET])
        ticket_scores.append(round(float(pipeline.predict_proba(ticket[FEATURES])[0, 1]), 3))
        all_scores.append(pipeline.predict_proba(valid[FEATURES])[:, 1])
    spread = pd.DataFrame(all_scores).std(ddof=0).mean()
    print(f"max_depth={depth}: T-117529 gets {ticket_scores}; mean spread over May {spread:.3f}")

> **Check.** With no limit, T-117529 gets `[0.0, 1.0, 0.0, 1.0, 0.0]` and the mean spread is `0.194`. With depth 3, it gets `[0.117, 0.099, 0.099, 0.12, 0.113]` and the mean spread is `0.024`.

The tree with no limit has **high variance**: its answer for the same ticket jumps between 0 and 1 with the train tickets. The depth-3 tree is stable, but it gives this escalated ticket only about 0.1: it has **more bias**. A good model sits between the two.

## 4. The depth sweep

The tree learns its questions and cut points from the data: these are its **parameters**. But you choose `max_depth` **before training**. A setting that you choose before training is a **hyperparameter**. You choose it like everything else in the loop: try several values and keep the best CV score. [Tune hyperparameters]({lesson_url("m06/tune-hyperparameters")}) in Module 6 does this for several at once. The next cell fits a tree for every `max_depth` from 1 to 20, and adds the gap: train AP − CV AP. It takes about 10 seconds.

> **Predict.** Which depth gives the best CV AP: 2, 6 or 20?

In [ ]:
rows = []
for depth in range(1, 21):
    result = evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=depth, random_state=0)))
    rows.append({"depth": depth, **result})
sweep = pd.DataFrame(rows).set_index("depth")
sweep["gap"] = (sweep["train AP"] - sweep["CV AP"]).round(3)
print(sweep.loc[[1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20]])

> **Check.** Depth 1: `0.178  0.178`. Depth 6: `0.346  0.258`, gap `0.088`. Depth 20: `0.996  0.139`, gap `0.857`.

Run the next cell to draw the two lines.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(sweep.index, sweep["train AP"], marker="o", color="C1", label="train AP (the tickets it learned from)")
ax.plot(sweep.index, sweep["CV AP"], marker="o", color="C0", label="CV AP (tickets it did not see)")
ax.axhline(0.117, linestyle="--", color="grey", label="one score for all (0.117)")
ax.set_xlabel("max_depth")
ax.set_ylabel("average precision")
ax.set_xticks(range(1, 21))
ax.legend()
plt.tight_layout()
plt.show()

> **Check.** The train line rises all the way to almost 1.0. The CV line rises to a peak at depth 6, then falls toward the floor.

Left of the peak, the tree underfits: both lines are low and close. Right of the peak, it overfits: the gap opens. The [explorer on the lesson page](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/#explore-the-depth) lets you move through the depths.

## 5. Guided practice: choose the depth with CV

> **Your turn.** From `sweep`, set `best_depth` to the depth with the highest CV AP, and `gap_at_best` to the gap at that depth. `sweep["CV AP"].idxmax()` gives the index of the largest value. Run the cell, then the check cell.

In [ ]:
# Your turn: from the table sweep, the depth with the highest CV AP, and the gap at that depth.
best_depth = int(sweep["CV AP"].idxmax())
gap_at_best = float(sweep.loc[best_depth, "gap"])
print(best_depth, gap_at_best)

In [ ]:
if best_depth is ... or gap_at_best is ...:
    print("Not yet: replace both ... with your code.")
else:
    expect('best_depth', best_depth, 6)
    expect('gap_at_best', gap_at_best, 0.088)

Even the best tree (0.258) stays far below logistic regression (0.327). Choosing the depth by CV, on train only, keeps the validation month free to confirm the choice later.

## 6. Your turn: limit the leaves instead of the depth

Change one condition. Keep no depth limit, and limit the size of a leaf instead: `min_samples_leaf` is the smallest number of train tickets that a leaf may have.

> **Predict.** Which value gives the best CV AP: 1, 20, 100 or 500? Will it beat the best depth (0.258)?

> **Your turn.** Replace `...` with `evaluate(tree_pipeline(DecisionTreeClassifier(min_samples_leaf=leaf, random_state=0)))`. Run the cell (about 10 seconds), then the check cell.

In [ ]:
leaf_rows = []
for leaf in [1, 5, 20, 50, 100, 200, 500]:
    result = evaluate(tree_pipeline(DecisionTreeClassifier(min_samples_leaf=leaf, random_state=0)))
    leaf_rows.append({"min_samples_leaf": leaf, **result})
leaf_sweep = pd.DataFrame(leaf_rows).set_index("min_samples_leaf")
print(leaf_sweep)

In [ ]:
if "CV AP" not in leaf_sweep:
    print("Not yet: replace ... with evaluate(tree_pipeline(DecisionTreeClassifier(...))), then run the cell again.")
else:
    best_leaf = int(leaf_sweep["CV AP"].idxmax())
    expect_hash("the best min_samples_leaf and its CV AP", [best_leaf, float(leaf_sweep.loc[best_leaf, "CV AP"])], "77bbc73c17cc149b")

Compare your table with the answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/#your-turn-limit-the-leaves-instead-of-the-depth).

## 7. Regularization in logistic regression

Logistic regression has a limit too. **Regularization** is a penalty on large weights, which keeps the model simpler. In scikit-learn, `C` sets it: a smaller `C` means a stronger penalty. The default is `C = 1.0`. Run the cell.

In [ ]:
strong_penalty = evaluate(build_pipeline(LogisticRegression(C=0.0001, max_iter=1000)))
print("logistic regression, C = 1 (default):", evaluate(build_pipeline()))
print("logistic regression, C = 0.0001:     ", strong_penalty)

> **Check.** With `C = 1`: train AP 0.328, CV AP 0.327. With `C = 0.0001`: train AP 0.301, CV AP 0.302.

A very strong penalty pushes all the weights toward zero, and the model underfits: both scores fall, and the gap stays near zero. At `C = 1`, logistic regression has almost no gap: it is not overfitting.

## 8. Learning curves

A **learning curve** shows the train and CV scores as the model gets more train tickets. The next cell computes it for three models, from 500 to 13,551 tickets (four fifths of train: the most that a fold allows). It takes about 5 seconds.

> **Predict.** If Larkfield collects twice as many tickets, which model gains the most: logistic regression, the depth-6 tree or the tree with no limit?

In [ ]:
sizes = [500, 1000, 2000, 4000, 8000, 13551]
curves = {}
for name, pipeline in {"logistic regression": build_pipeline(),
                       "tree, depth 6": tree_pipeline(DecisionTreeClassifier(max_depth=6, random_state=0)),
                       "tree, no limit": tree_pipeline(DecisionTreeClassifier(random_state=0))}.items():
    n, train_scores, cv_scores = learning_curve(pipeline, train[FEATURES], train[TARGET], cv=cv, train_sizes=sizes,
                                                scoring="average_precision", shuffle=True, random_state=0)
    curves[name] = (train_scores.mean(axis=1), cv_scores.mean(axis=1))
    print(f"{name:20s} train {train_scores.mean(axis=1).round(3)}")
    print(f"{'':20s} CV    {cv_scores.mean(axis=1).round(3)}")

> **Check.** Logistic regression: train falls from 0.408 to 0.329, CV rises from 0.279 to 0.327. Depth 6: train falls from 0.681 to 0.355, CV rises from 0.157 to 0.258. No limit: train 1.0 at every size, CV between 0.135 and 0.141.

Run the next cell to draw the curves.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for (name, (train_line, cv_line)), color in zip(curves.items(), ["C0", "C1", "C2"]):
    ax.plot(sizes, train_line, linestyle="--", color=color, label=f"{name}, train")
    ax.plot(sizes, cv_line, marker="o", color=color, label=f"{name}, CV")
ax.set_xlabel("train tickets used to fit")
ax.set_ylabel("average precision")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

> **Check.** For logistic regression, the dashed train line and the solid CV line almost meet. For the depth-6 tree, they are still apart and coming closer. For the tree with no limit, they stay far apart.

- **Logistic regression:** the lines have met, and the CV line is almost flat. More tickets of the same kind would help little. To do better, the model needs better features (Module 6).
- **Depth 6:** the CV line still rises. More tickets would help this tree.
- **No limit:** more tickets do not help. The tree memorises any number of tickets.

## 9. Update the experiment log

The best tree so far (`min_samples_leaf=100`) and the strong penalty (`C = 0.0001`) go in the log. Neither beats E03.

> **Your turn.** Write the rows E09 and E10 with `log_experiment()`: the change, the CV AP, the std, `None` for the valid AP, the decision `"reject"` and a reason. Run the cell, then the check cell.

In [ ]:
# Your turn: write the rows E09 (the tree with min_samples_leaf=100) and E10 (C=0.0001).
log_experiment("E09", "Decision tree, no depth limit, min_samples_leaf=100", leaf_sweep.loc[100, "CV AP"],
               leaf_sweep.loc[100, "std"], None, "reject", "The best tree so far, still far below E03.")
log_experiment("E10", "E03 with C=0.0001 (a very strong penalty)", strong_penalty["CV AP"], strong_penalty["std"],
               None, "reject", "It underfits: both train and CV AP fall.")
show_log(columns=("id", "change", "cv_ap", "cv_std", "decision"))

In [ ]:
log = pd.read_csv(LOG_FILE).set_index("id")
missing = [row_id for row_id in ['E09', 'E10'] if row_id not in log.index]
if missing:
    print("Not yet: the log is missing", ", ".join(missing) + ". Write the rows with log_experiment(), then run this cell again.")
else:
    rows = log.loc[['E09', 'E10']]
    expect_hash("the log from E09 to E10", [[r.cv_ap, r.cv_std, None if pd.isna(r.valid_ap) else r.valid_ap, r.decision]
                                   for r in rows.itertuples()], "a92a581bf78707e1")

> **Check.** You should see ten rows, and `Check passed: the log from E09 to E10 is right.` E03 is still the best.

## 10. Failure case: no gap, so no problem?

Tomás wants a model with no overfitting at all. Run the cell.

In [ ]:
print("tree, depth 1:", evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=1, random_state=0))))
print("tree, depth 6:", evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=6, random_state=0))))

> **Check.** Depth 1: `train AP 0.178`, `CV AP 0.178`. Depth 6: `0.346` and `0.258`.

The depth-1 tree has no gap. But its CV AP, 0.178, is only a little above the floor of 0.117, and far below the depth-6 tree. **A small gap does not mean a good model.** Look at the level too: compare the CV score with the baseline and with the best model. The [lesson page](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/#no-gap-so-no-problem) has the full failure case.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you saw underfitting (both scores low) and overfitting (a large gap), measured variance on five halves of train, chose `max_depth` and `min_samples_leaf` by CV, saw what `C` does, read learning curves, and logged two more experiments. Next, you combine many trees into one model, and see what that costs.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Ensembles](https://learning.nextia-ai.com/courses/ml/m05/ensembles/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.